# Lean4 / Dragapult — BC initialization for PPO (Kaggle)

Input:
- `lean4_bc_dataset.npz` (produced from the existing replay corpus)
- this repository as a code dataset (optional; the notebook is self-contained)

This is **stage 1**: behavior cloning on existing lean4 decisions. It is the initialization for the PPO/league stage; it is not the final PPO run.


In [ ]:
from pathlib import Path
import json, math, random, time
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

INPUT = Path("/kaggle/input")
DATA = next(ITER := iter(INPUT.rglob("lean4_bc_dataset.npz")), None) if INPUT.exists() else Path("data/lean4_bc_dataset.npz")
assert DATA and DATA.is_file(), f"dataset not found: {DATA}"
WORK = Path("/kaggle/working"); WORK.mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("dataset", DATA, DATA.stat().st_size/1e6, "MB", "device", device)


In [ ]:
z = np.load(DATA)
state = torch.from_numpy(z["state"].astype(np.float32))
options = torch.from_numpy(z["options"].astype(np.float32))
mask = torch.from_numpy(z["mask"].astype(np.float32))
card_ids = torch.from_numpy(z["card_ids"].astype(np.int64))
label = torch.from_numpy(z["label"].astype(np.int64))
split = torch.from_numpy(z["split"].astype(np.int8))
tr = split == 0; va = split == 1
print("decisions", len(label), "train", int(tr.sum()), "val", int(va.sum()))
print("state", tuple(state.shape), "options", tuple(options.shape), "max card", int(card_ids.max()))


In [ ]:
class OptionPolicy(nn.Module):
    def __init__(self, state_dim=50, option_dim=48, card_vocab=4096, emb=32, hidden=128):
        super().__init__()
        self.card = nn.Embedding(card_vocab, emb, padding_idx=0)
        self.state = nn.Sequential(nn.Linear(state_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.opt = nn.Sequential(nn.Linear(option_dim + 2*emb, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.head = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, s, o, ids):
        a = self.card(ids[...,0]); b = self.card(ids[...,1])
        ot = self.opt(torch.cat([o, a, b], dim=-1))
        st = self.state(s).unsqueeze(1).expand(-1, ot.size(1), -1)
        return self.head(torch.cat([ot, st], dim=-1)).squeeze(-1)

net = OptionPolicy().to(device)
print("params", sum(p.numel() for p in net.parameters()))
opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=1e-4)


In [ ]:
BATCH=512
train_ds = TensorDataset(state[tr], options[tr], mask[tr], card_ids[tr], label[tr])
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
val_s,val_o,val_m,val_c,val_y = state[va],options[va],mask[va],card_ids[va],label[va]
lossfn = nn.CrossEntropyLoss()
for epoch in range(1, 6):
    net.train(); total=0; n=0
    for s,o,m,c,y in train_dl:
        s,o,m,c,y = s.to(device),o.to(device),m.to(device),c.to(device),y.to(device)
        logits = net(s,o,c).masked_fill(m==0, -1e9)
        loss = lossfn(logits, y)
        opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
        total += loss.item()*len(y); n += len(y)
    net.eval(); correct=0; totalv=0
    with torch.no_grad():
        for i in range(0,len(val_y),1024):
            sl=slice(i,i+1024)
            logits=net(val_s[sl].to(device),val_o[sl].to(device),val_c[sl].to(device)).masked_fill(val_m[sl].to(device)==0,-1e9)
            correct += int((logits.argmax(-1).cpu()==val_y[sl]).sum()); totalv += len(val_y[sl])
    print(f"epoch {epoch} loss {total/max(1,n):.4f} val_top1 {correct/max(1,totalv):.4f}", flush=True)
torch.save(net.state_dict(), WORK/"lean4_bc_model.pt")
print("saved", WORK/"lean4_bc_model.pt")


In [ ]:
# Optional: inspect per-decision top-3 accuracy and export feature metadata
net.eval(); top3=0; total=0
with torch.no_grad():
    for i in range(0,len(val_y),1024):
        sl=slice(i,i+1024)
        logits=net(val_s[sl].to(device),val_o[sl].to(device),val_c[sl].to(device)).masked_fill(val_m[sl].to(device)==0,-1e9)
        top=logits.topk(3,dim=-1).indices.cpu(); y=val_y[sl]
        top3 += int((top==y[:,None]).any(dim=1).sum()); total += len(y)
print("val top3", top3/max(1,total))
print("next: use this checkpoint as the BC initialization for the PPO/league stage")
